In [ ]:
# %reset -f
%load_ext autoreload
%autoreload complete --log


In [ ]:
from smtgraphformer import *
from smtgraphformer.contextEncoding import TransformBundle, configCE
from smtgraphformer.smtGraphFormer import *

setDisplayOptions()
sr = setReproducibility(17711)


### Load Saved Artefacts

In [ ]:
root = Path("../data")
# ---
bundle = TransformBundle.load(f"{root}/AtB-TransformBundle.json")
ds_splits = pd.read_csv(f"{root}/AtB-SplitPlan.csv")
sattrs = pd.read_csv(f"{root}/AtB-StopAttributes.csv")
lattrs = pd.read_csv(f"{root}/AtB-LineAttributes.csv")
# ---
tripsSMT = pd.read_pickle(f"{root}/AtB-tfmTripLevelSMT.pkl")
mobilityPatterns = pklLoader(f"{root}/AtB-MobilityPatterns-Train.pkl")
stoprms = np.load(f"{root}/AtB-buildRelationalMatrices.npy")


In [ ]:
assert stoprms.ndim in (3, 4), "!!!"
assert stoprms.shape[-1] == stoprms.shape[-2], "!!!"
assert ds_splits["$split"].isin(["train", "valid", "test"]).all(), "!!!"
# ---
print(f"{tripsSMT.shape=}")
print(f"{sattrs.shape=}, {lattrs.shape=}")
print(f"{stoprms.shape=}")
print(f"split counts={tripsSMT.groupby('$split', observed=True).size().to_dict()}")
print(f"targets={list(bundle.targetTransforms)}")


### Model Config and Dataloader Setup

In [ ]:
cfgModel = SMTConfig(
    embed_dim=256,
    batch_size=256,
    n_head=8,  # 4/8/12
    n_layer=4,  # 4/8/12
    n_expert=3,  # 3/5
    # training
    train_epochs=40,
    early_stopping="valid",
    early_stopping_patience=20,
    pct_warmup=0.075,
    scheduled_sampling=False,
    ss_max_probability=0.5,
    # optimisation
    w_surrogate_tasks=1.0,
    learning_rate=3e-4,
    dropout_pct=0.1,
    max_grad_norm=1.0,
    w_decay=0.01,
    # saving
    model_dir="../models",
    model_name="smtM24",
)

# init dataloader; update config with dataloader-derived values
dls = SMTDataloader(tripsSMT, cfg=cfgModel, bundle=bundle, sattrs=sattrs)
cfgModel.vocab_size = dls.vocab_size
cfgModel.ctx_length = dls.ctx_length
cfgModel.eval_interval = estimateEvalInterval(dls, cfgModel, pct=1)

# build context encoder config from dataloader attributes
cfgContext = configCE(
    lc_cats=dls.ce_lccats,
    conts=dls.ce_nconts,
    output=cfgModel.trip_context_dim,  # type:ignore
)


In [ ]:
print(f"{dls.info()=}")
print(f"special tokens: {dls.stoi['<SOS>']=}, {dls.stoi['<EOS>']=}, {dls.stoi['<PAD>']=}")
print(f"context: ce_lccats={dls.ce_lccats}, ce_nconts={dls.ce_nconts}")


### Train Graph Autoencoder


In [ ]:
# stack all matrices; add dummy batch dimension > [1, n_matrices, n_nodes, n_nodes]
t_stoprms = torch.tensor(stoprms, dtype=torch.float32)
t_stoprms = t_stoprms.unsqueeze(0) if stoprms.ndim == 3 else t_stoprms
print(f" > stacked matrices tensor: {t_stoprms.shape}")

gaeModel, gaeEmbeddings = trainGAEModelEpochs(t_stoprms, cfg=cfgModel, n_epochs=4096)

lfp = lambda name: f"{name:>20}:"  # left format print
print(lfp("graph embeddings"), f"{gaeEmbeddings.shape}")
print(lfp("vocabulary size"), f"{gaeEmbeddings.shape[0]}")
print(lfp("embedding dimension"), f"{gaeEmbeddings.shape[1]}")


### Model Initialisation & Training

In [ ]:
# initialize model with graph embeddings as frozen buffer
model = SMTGraphFormer(
    cfg=cfgModel,
    cfgContext=cfgContext,
    graph_embeddings=gaeEmbeddings,
    stop_features=dls.stopFeatures,
)
model.dryrun(dls)


In [ ]:
params = {"m": model, "dls": dls, "cfg": cfgModel, "save_model": True, "final_eval": True}
trainer = trainSMTScheduledSampling if cfgModel.scheduled_sampling else trainSMTModelEpochs
log_metrics = trainer(**params)


In [ ]:
stats = saveTrainingSummary(model, cfgModel, plot=False, stats=True, verbose=True)
fp = f"{model.model_dir}/{model.model_tag}/history.png"
plotTrainingHistory(log_metrics, fp=fp)


### Evaluation Example

In [ ]:
# tag = "smtM24-123456789"  # replace with actual tag
# cfgModel.model_dir = "../models/"
# gaeEmbeddings = np.load(f"{cfgModel.model_dir}/{tag}/gaeEmbeddings.npy")
# gaeEmbeddings = torch.tensor(gaeEmbeddings, dtype=torch.float32).to(cfgModel.device)

# model = SMTGraphFormer(
#     cfg=cfgModel,
#     cfgContext=cfgContext,
#     graph_embeddings=gaeEmbeddings,
#     stop_features=dls.stopFeatures,
# )
# model.load(tag=tag)

# fp = f"{cfgModel.model_dir}/{tag}/training.log"
# log_metrics = loadTrainingLog(fp)


In [ ]:
teacher_forcing = True

l_metrics = []
comparisons = {}
for split in ["train", "valid", "test"]:
    s_metrics, s_comparison = smtFinalEvaluation(
        model, dls, bundle, split, raw_evaluation=True, teacher_forcing=teacher_forcing
    )
    s_metrics.insert(0, "$split", split)
    l_metrics.append(s_metrics)
    comparisons[split] = s_comparison

metrics = pd.concat(l_metrics, ignore_index=True)
print(metrics.tail(4))


In [ ]:
suffix = "forced" if teacher_forcing else "autoreg"
df_csver(metrics, tag=f"{model.model_dir}/{model.model_tag}/metrics.{suffix}")


### end